# Structural Decay of Cross-Factor Predictability
## ICAIF 2026 — Reproducibility Notebook

This notebook serves as a reproducibility verification for the paper "Structural Decay of Cross-Factor Predictability."

**Important:** This notebook loads and verifies pre-computed results against paper claims. It does NOT re-run the experiments, which require significant computational resources. Each experiment is implemented as an independent script and pre-computed results are stored in JSON format.

The notebook will:
1. Load all pre-computed result files
2. Extract key statistics from each experiment
3. Compare against reported paper values
4. Print PASS/FAIL status for each table/claim
5. Provide instructions for reproducing results from scratch if needed

In [ ]:
import json
import os
from pathlib import Path

# Set up paths
repo_root = Path.cwd()
results_dir = repo_root / "results"

# Dictionary to store all loaded results
results = {}

# List of result files to load
result_files = [
    "three_fold_validation.json",
    "macro_regime_granger.json",
    "holdings_deleveraging.json",
    "emerging_market_validation.json",
    "neural_granger_selected.json",
    "lstm_granger_results.json",
    "risk_monitoring_results.json",
    "nonfinance_domain_validation.json",
]

# Load all result files
print("=" * 80)
print("LOADING PRE-COMPUTED RESULTS")
print("=" * 80)

for filename in result_files:
    filepath = results_dir / filename
    try:
        with open(filepath, 'r') as f:
            results[filename.replace('.json', '')] = json.load(f)
        print(f"✓ Loaded {filename}")
    except FileNotFoundError:
        print(f"✗ NOT FOUND: {filename}")
    except json.JSONDecodeError as e:
        print(f"✗ ERROR reading {filename}: {e}")

print("=" * 80)
print(f"Successfully loaded {len(results)} result files\n")

## Table 3: Granger Predictability HML-SMB

**Primary In-Sample Result:** The main pipeline produces an in-sample Granger causality p-value of **p = 8.75e-9** for HML → SMB predictability over the full sample period.

This is the core finding of the paper's main analysis. The detailed pipeline involves:
- Rolling-window Granger tests with causal regime detection
- HAC (Heteroskedasticity and Autocorrelation Consistent) standard errors
- Regime-conditional Wald statistics
- Cross-validation on held-out data

This result is not re-computed in this notebook as it requires the full data pipeline. Instead, this notebook verifies supporting results from different domains and validation approaches.

In [ ]:
print("=" * 80)
print("TABLE 5 VERIFICATION: FOUR-MODEL DIAGNOSTIC (HML→SMB)")
print("=" * 80)

# Table 5 uses neural_granger_selected.json (selected HMM fit)
ng = results.get('neural_granger_selected', {})
table5_pass = True

if ng:
    rd = ng.get('results', {})
    # Paper values (Table 5, line 441-443 of main_icaif.tex):
    #   Normal:   n=4,496, Linear 0.86%**, RF p=0.69, MLP p=0.20, LSTM p=0.63
    #   Elevated: n=2,792, Linear 0.92%**, RF p=1.00, MLP p=1.00, LSTM p=0.93
    #   Crisis:   n=1,017, Linear 0.43%,   RF p=0.13, MLP p=0.96, LSTM p=0.83
    
    expected = {
        'Normal':   {'lin': 0.86, 'rf_p': 0.69, 'mlp_p': 0.20, 'n': 4496},
        'Elevated': {'lin': 0.92, 'rf_p': 1.00, 'mlp_p': 1.00, 'n': 2792},
        'Crisis':   {'lin': 0.43, 'rf_p': 0.13, 'mlp_p': 0.96, 'n': 1017},
    }
    
    for regime in ['Normal', 'Elevated', 'Crisis']:
        d = rd.get(regime, {})
        lin_pct = d.get('linear_mse_improvement_pct', None)
        rf_p = d.get('rf_p_value', None)
        mlp_p = d.get('mlp_p_value', None)
        n_clean = d.get('n_clean', None)
        
        exp = expected[regime]
        print(f"\n  {regime} (n={n_clean}, paper: {exp['n']}):")
        
        if n_clean is not None:
            ok_n = n_clean == exp['n']
            print(f"    Sample size: {n_clean}  (paper: {exp['n']})  {'PASS' if ok_n else 'FAIL'}")
            if not ok_n: table5_pass = False
        
        if lin_pct is not None:
            ok = abs(lin_pct - exp['lin']) < 0.05
            print(f"    Linear improvement: {lin_pct:.2f}%  (paper: {exp['lin']:.2f}%)  {'PASS' if ok else 'FAIL'}")
            if not ok: table5_pass = False
        else:
            print(f"    Linear improvement: MISSING"); table5_pass = False
        
        if rf_p is not None:
            ok = abs(rf_p - exp['rf_p']) < 0.02
            print(f"    RF p-value: {rf_p:.3f}  (paper: {exp['rf_p']:.2f})  {'PASS' if ok else 'FAIL'}")
            if not ok: table5_pass = False
        
        if mlp_p is not None:
            ok = abs(mlp_p - exp['mlp_p']) < 0.05
            print(f"    MLP p-value: {mlp_p:.3f}  (paper: {exp['mlp_p']:.2f})  {'PASS' if ok else 'FAIL'}")
            if not ok: table5_pass = False

    print(f"\n{'='*80}")
    print(f"TABLE 5 RESULT: {'PASS' if table5_pass else 'FAIL'}")
else:
    print("ERROR: neural_granger_selected not loaded")
    table5_pass = False

In [ ]:
print("=" * 80)
print("THREE-FOLD TEMPORAL VALIDATION")
print("=" * 80)

tf = results.get('three_fold_validation', {})
threefold_pass = True

if tf:
    folds = tf.get('folds', {})
    for fold_key, fold_label in [('fold_b', 'Fold B (2001-2012)'), ('fold_c', 'Fold C (2013-2024)')]:
        fold = folds.get(fold_key, {})
        granger = fold.get('granger_hml_to_smb', {})
        print(f"\n  {fold_label}:")
        
        for regime in ['Normal', 'Elevated', 'Crisis']:
            g = granger.get(regime, {})
            hac_p = g.get('hac_p_value', None)
            f_p = g.get('f_p_value', None)
            n_obs = g.get('n_obs', '?')
            
            if hac_p is not None:
                ok = hac_p > 0.10
                print(f"    {regime} (n={n_obs}): HAC p={hac_p:.4f}  (>0.10? {'PASS' if ok else 'FAIL'})")
                if not ok: threefold_pass = False
            elif f_p is not None:
                ok = f_p > 0.10
                print(f"    {regime} (n={n_obs}): F p={f_p:.4f}  (>0.10? {'PASS' if ok else 'FAIL'})")
                if not ok: threefold_pass = False
            else:
                print(f"    {regime}: insufficient data (expected null)")

    print(f"\n{'='*80}")
    print(f"THREE-FOLD RESULT: {'PASS' if threefold_pass else 'FAIL'}")
    print("  Paper claim: 'All folds null, p > 0.10' — ", "CONFIRMED" if threefold_pass else "REJECTED")
else:
    print("ERROR: three_fold_validation not loaded")
    threefold_pass = False

In [ ]:
print("=" * 80)
print("MACRO DOMAIN: T10Y2Y → INDPRO (Crisis regime, test period 2006-2015)")
print("=" * 80)

mg = results.get('macro_regime_granger', {})
macro_pass = True

if mg:
    crisis = mg.get('test', {}).get('granger_t10y2y_to_indpro', {}).get('Crisis', None)
    
    if crisis:
        f_stat = crisis['f_stat']
        hac_p = crisis['hac_p_value']
        dr2 = crisis['delta_r2']
        n = crisis['n_obs']
        
        # Paper claims: F=12.7, HAC p=0.0002, ΔR²=11.9%
        f_ok = abs(f_stat - 12.7) < 1.0
        p_ok = hac_p < 0.001
        r2_ok = abs(dr2 - 0.119) < 0.01
        
        print(f"\n  Crisis regime (n={n}):")
        print(f"    F-stat:  {f_stat:.2f}   (paper: 12.7)     {'PASS' if f_ok else 'FAIL'}")
        print(f"    HAC p:   {hac_p:.6f} (paper: 0.0002)  {'PASS' if p_ok else 'FAIL'}")
        print(f"    ΔR²:     {dr2:.4f}  ({dr2*100:.1f}%, paper: 11.9%)  {'PASS' if r2_ok else 'FAIL'}")
        
        macro_pass = f_ok and p_ok and r2_ok
    else:
        print("  Crisis regime: no result (Normal/Elevated had insufficient obs)")
        macro_pass = False
    
    # Also check OOS is null (disclosure)
    oos = mg.get('oos', {}).get('granger_t10y2y_to_indpro', {})
    all_null = all(v is None for v in oos.values())
    print(f"\n  OOS (2016-2024): {'all null (insufficient obs)' if all_null else 'has results'}")
    
    print(f"\n{'='*80}")
    print(f"MACRO RESULT: {'PASS' if macro_pass else 'FAIL'}")
else:
    print("ERROR: macro_regime_granger not loaded")
    macro_pass = False

In [ ]:
print("=" * 80)
print("HOLDINGS-BASED DELEVERAGING MECHANISM")
print("=" * 80)

hd = results.get('holdings_deleveraging', {})
holdings_pass = True

if hd:
    gr = hd.get('granger_results', {})
    crowding_split = gr.get('hml_to_smb_normal_regime_by_crowding', {})
    
    # Paper: high-crowding F=159.5, ΔR²=4.4%; low F=139.2, ΔR²=3.8%
    high = crowding_split.get('high_crowding', {})
    low = crowding_split.get('low_crowding', {})
    
    if high and low:
        hf, hdr2 = high['f_stat'], high['delta_r2']
        lf, ldr2 = low['f_stat'], low['delta_r2']
        pct_larger = (hdr2 - ldr2) / ldr2 * 100
        
        print(f"\n  High crowding: F={hf:.1f}, ΔR²={hdr2*100:.1f}%  (paper: F=159.5, 4.4%)")
        print(f"  Low crowding:  F={lf:.1f}, ΔR²={ldr2*100:.1f}%  (paper: F=139.2, 3.8%)")
        print(f"  ΔR² ratio: {pct_larger:.0f}% larger  (paper: 14%)")
        
        ok_hf = abs(hf - 159.5) < 5
        ok_lf = abs(lf - 139.2) < 5
        ok_ratio = abs(pct_larger - 14) < 5
        
        print(f"  High F: {'PASS' if ok_hf else 'FAIL'}  Low F: {'PASS' if ok_lf else 'FAIL'}  Ratio: {'PASS' if ok_ratio else 'FAIL'}")
        if not (ok_hf and ok_lf and ok_ratio): holdings_pass = False
    else:
        print("  ERROR: crowding split data missing"); holdings_pass = False
    
    # Spearman
    spa = hd.get('portfolio_level_analysis', {}).get('rank_correlation_overlap_vs_signal', {})
    if spa:
        rho = spa['rho_spearman']
        sp_p = spa['p_value']
        sp_n = spa['n_portfolios']
        rho_ok = abs(rho - 0.51) < 0.05
        p_ok = abs(sp_p - 0.009) < 0.005
        print(f"\n  Spearman ρ={rho:.2f}, p={sp_p:.4f}, n={sp_n}  (paper: ρ=0.51, p=0.009, n=25)")
        print(f"  ρ: {'PASS' if rho_ok else 'FAIL'}  p: {'PASS' if p_ok else 'FAIL'}")
        if not (rho_ok and p_ok): holdings_pass = False
    
    # Crowding→SMB per regime
    cg = gr.get('crowding_to_smb_by_regime', {})
    if cg:
        print(f"\n  Crowding→SMB Granger (paper: p>0.60 all regimes):")
        for regime, res in cg.items():
            if res:
                p_val = res['f_p_value']
                ok = p_val > 0.60
                print(f"    {regime}: p={p_val:.3f}  {'PASS' if ok else 'FAIL'}")
                if not ok: holdings_pass = False
    
    print(f"\n{'='*80}")
    print(f"HOLDINGS RESULT: {'PASS' if holdings_pass else 'FAIL'}")
else:
    print("ERROR: holdings_deleveraging not loaded")
    holdings_pass = False

In [ ]:
print("=" * 80)
print("EMERGING MARKET VALIDATION")
print("=" * 80)

em = results.get('emerging_market_validation', {})
em_pass = True

if em:
    pr = em.get('pre_registration', {})
    n_pairs = pr.get('n_pairs_tested', None)
    bonf = pr.get('bonferroni_alpha', None)
    sig = pr.get('significant_pairs_found', None)
    
    # Paper: 20 pairs, α=0.0025, 6 significant
    print(f"\n  Pairs tested:    {n_pairs}  (paper: 20)   {'PASS' if n_pairs==20 else 'FAIL'}")
    print(f"  Bonferroni α:    {bonf}  (paper: 0.0025)  {'PASS' if bonf==0.0025 else 'FAIL'}")
    print(f"  Significant:     {sig}  (paper: 6)       {'PASS' if sig==6 else 'FAIL'}")
    
    if n_pairs != 20 or bonf != 0.0025 or sig != 6:
        em_pass = False
    
    print(f"\n{'='*80}")
    print(f"EMERGING MARKET RESULT: {'PASS' if em_pass else 'FAIL'}")
else:
    print("ERROR: emerging_market_validation not loaded")
    em_pass = False

In [ ]:
print("=" * 80)
print("VaR RISK MONITORING APPLICATION")
print("=" * 80)

rk = results.get('risk_monitoring_results', {})
var_pass = True

if rk:
    overall = rk.get('overall_results', {})
    unc = overall.get('unconditional_historical', {})
    hml = overall.get('hml_informed', {})
    
    unc_dev = unc.get('deviation_from_target_pct', None)
    hml_dev = hml.get('deviation_from_target_pct', None)
    unc_cc = unc.get('christoffersen_p_cc', None)
    hml_cc = hml.get('christoffersen_p_cc', None)
    
    # Paper: unconditional +1.39pp, HML-informed +0.82pp
    # Christoffersen: unconditional p=0.002, HML p=0.115
    print(f"\n  Unconditional deviation: +{unc_dev:.2f}pp  (paper: +1.39pp)  {'PASS' if abs(unc_dev-1.39)<0.1 else 'FAIL'}")
    print(f"  HML-informed deviation:  +{hml_dev:.2f}pp  (paper: +0.82pp)  {'PASS' if abs(hml_dev-0.82)<0.1 else 'FAIL'}")
    print(f"  Christoffersen p (unc):  {unc_cc:.4f}  (paper: 0.002)  {'PASS' if unc_cc<0.01 else 'FAIL'}")
    print(f"  Christoffersen p (HML):  {hml_cc:.4f}  (paper: 0.115)  {'PASS' if abs(hml_cc-0.115)<0.02 else 'FAIL'}")
    
    if abs(unc_dev-1.39)>0.1 or abs(hml_dev-0.82)>0.1 or unc_cc>0.01 or abs(hml_cc-0.115)>0.02:
        var_pass = False
    
    print(f"\n{'='*80}")
    print(f"VaR RESULT: {'PASS' if var_pass else 'FAIL'}")
else:
    print("ERROR: risk_monitoring_results not loaded")
    var_pass = False

In [ ]:
print("=" * 80)
print("SYNTHETIC ENSO (NON-FINANCE DOMAIN VALIDATION)")
print("=" * 80)

nf = results.get('nonfinance_domain_validation', {})
enso_pass = True

if nf:
    climate = nf.get('domains', {}).get('climate_teleconnections', {})
    
    if climate:
        ari = climate.get('hmm', {}).get('adjusted_rand_index', None)
        
        # Causal regime (regime 2) ΔR²
        gr = climate.get('granger_results', {})
        causal_dr2 = gr.get('2', {}).get('delta_r2', None)
        
        # Unconditional ΔR²
        unc_dr2 = climate.get('unconditional_granger', {}).get('delta_r2', None)
        
        T = climate.get('ground_truth', {}).get('T', None)
        
        # Paper: ARI=0.72, causal ΔR²=25.8%, unconditional=14.2%, T=2000
        print(f"\n  ARI:               {ari:.4f}  (paper: 0.72)    {'PASS' if abs(ari-0.72)<0.02 else 'FAIL'}")
        print(f"  Causal regime ΔR²: {causal_dr2*100:.1f}%  (paper: 25.8%)  {'PASS' if abs(causal_dr2-0.258)<0.01 else 'FAIL'}")
        print(f"  Unconditional ΔR²: {unc_dr2*100:.1f}%  (paper: 14.2%)  {'PASS' if abs(unc_dr2-0.142)<0.01 else 'FAIL'}")
        print(f"  T:                 {T}    (paper: 2000)   {'PASS' if T==2000 else 'FAIL'}")
        
        # Ground truth recovery
        gt = climate.get('ground_truth_recovery', {})
        causal_detected = gt.get('causal_regime_detected', False)
        non_causal_masked = gt.get('non_causal_regimes_masked', False)
        print(f"  Causal regime detected:    {causal_detected}  {'PASS' if causal_detected else 'FAIL'}")
        print(f"  Non-causal regimes masked: {non_causal_masked}  {'PASS' if non_causal_masked else 'FAIL'}")
        
        if (abs(ari-0.72)>0.02 or abs(causal_dr2-0.258)>0.01 or 
            abs(unc_dr2-0.142)>0.01 or T!=2000 or 
            not causal_detected or not non_causal_masked):
            enso_pass = False
    else:
        print("  ERROR: climate_teleconnections data missing")
        enso_pass = False
    
    print(f"\n{'='*80}")
    print(f"ENSO RESULT: {'PASS' if enso_pass else 'FAIL'}")
else:
    print("ERROR: nonfinance_domain_validation not loaded")
    enso_pass = False

In [ ]:
print("\n" + "=" * 80)
print("FINAL REPRODUCIBILITY SUMMARY")
print("=" * 80)

# Collect actual pass/fail from each cell's variable
validation_results = {
    'Table 4 (Four-Model Diagnostic)': table5_pass,
    'Table 5 (Three-Fold Validation)': threefold_pass,
    'Table 6 (Macro Domain)': macro_pass,
    'Table 7 (Holdings Deleveraging)': holdings_pass,
    'Table 8 (Emerging Markets)': em_pass,
    'Table 9 (VaR Risk Application)': var_pass,
    'Table 10 (Synthetic ENSO)': enso_pass,
}

print("\nValidation Status by Table:")
print("-" * 80)

total_tables = len(validation_results)
passed_tables = sum(1 for v in validation_results.values() if v)

for table_name, status in validation_results.items():
    status_str = 'PASS' if status else 'FAIL'
    symbol = '✓' if status else '✗'
    print(f"  {symbol} {table_name}: {status_str}")

print("\n" + "=" * 80)
print(f"FINAL RESULT: {passed_tables}/{total_tables} TABLES VERIFIED")
print("=" * 80)

if passed_tables == total_tables:
    print("\nAll results successfully verified against paper claims!")
    print("The pre-computed results are consistent with the published findings.")
else:
    print(f"\nWarning: {total_tables - passed_tables} validation(s) failed.")
    print("See detailed checks above for specific discrepancies.")

## How to Reproduce Results from Scratch

This notebook verifies pre-computed results. If you wish to reproduce the experiments independently, each analysis is implemented as a standalone Python script in the `code/` directory.

### Individual Reproducibility Scripts

Each script can be executed independently and will generate its corresponding result file:

#### 1. **Three-Fold Temporal Validation**
```bash
python code/three_fold_validation.py
```
- **Output:** `results/three_fold_validation.json`
- **Purpose:** Tests temporal stability of HML→SMB predictability across three non-overlapping folds
- **Validates:** Out-of-sample Granger causality p-values > 0.10 (no spurious predictability)

#### 2. **Macro Domain Test (T10Y2Y → INDPRO)**
```bash
python code/macro_regime_granger.py
```
- **Output:** `results/macro_regime_granger.json`
- **Purpose:** Demonstrates causal regime detection in macroeconomic data
- **Validates:** Crisis regime F-stat ≈ 12.7, p ≈ 0.0002, ΔR² ≈ 11.9%

#### 3. **Holdings-Based Deleveraging Analysis**
```bash
python code/holdings_deleveraging.py
```
- **Output:** `results/holdings_deleveraging.json`
- **Purpose:** Tests crowding-driven deleveraging as a mechanism driving SMB returns
- **Validates:** High-crowding portfolio predictability, Spearman ρ, regime-conditional Granger tests

#### 4. **Emerging Market Validation**
```bash
python code/emerging_market_validation.py
```
- **Output:** `results/emerging_market_validation.json`
- **Purpose:** Cross-country validation of causal structure
- **Validates:** International evidence for factor causality with Bonferroni correction

#### 5. **Non-Finance Domain Validation (Synthetic ENSO)**
```bash
python code/nonfinance_domain_validation.py
```
- **Output:** `results/nonfinance_domain_validation.json`
- **Purpose:** Demonstrates causal regime detection on synthetic El Niño data
- **Validates:** Method generalizability beyond financial data

### Computational Requirements

- **Three-Fold Validation:** ~2-4 hours
- **Macro Granger:** ~30 minutes
- **Holdings Deleveraging:** ~1-2 hours
- **Emerging Markets:** ~3-5 hours
- **Non-Finance Domain:** ~15 minutes
- **Total:** ~6-13 hours depending on system

### Data Requirements

All scripts load data from standard sources:
- **Fama-French Factors:** Downloaded via `pandas_datareader`
- **Macro Data:** FRED API (requires free API key)
- **Holdings Data:** Processed from public SEC EDGAR filings
- **EM Data:** Public asset pricing databases

### Dependencies

Core requirements are specified in `requirements.txt`:
```
numpy>=1.20
scipy>=1.7
pandas>=1.3
statsmodels>=0.13
scikit-learn>=1.0
```

Install with:
```bash
pip install -r requirements.txt
```

### Verification Workflow

After running scripts (or using pre-computed results):
1. Run this notebook cell-by-cell
2. Compare outputs against expected paper values
3. Check PASS/FAIL status in each table cell
4. Review final summary for overall reproducibility status